## 1. Mount Drive, settings and paths

In [ ]:
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

RUN_ID = "gee_20261109"
YEARS = "2025-2026"
REPO_SOURCE = "git"
REPO_URL = "https://github.com/farzadasgari/rivwq.git"
PYTHON_VERSION = "3.13"
EE_AUTH_MODE = None
SESSION_LIMIT_HOURS = 12
STOP_MARGIN_MINUTES = 15
STATUS_EVERY_MINUTES = 5
SYNC_EVERY_MINUTES = 10
MONITOR_TICK_SECONDS = 20
RESTART_WAIT_SECONDS = 300
MAX_RESTARTS = 20
STOP_TIMEOUT_SECONDS = 120
ALLOW_COMMIT_CHANGE = False
ALLOW_BAD_FILES = False

MOUNT = Path("/content/drive")
WORK = Path("/content")
DRIVE_DIR = MOUNT / "MyDrive" / "rivwq_colab"
DRIVE_RUN = DRIVE_DIR / "run" / RUN_ID
DRIVE_LOGS = DRIVE_DIR / "logs"
RUN_ZIP = DRIVE_DIR / f"{RUN_ID}.zip"
REPO_ZIP = DRIVE_DIR / "rivwq.zip"
COMMIT_RECORD = DRIVE_DIR / f"{RUN_ID}_commit.txt"
REPO = WORK / "rivwq"
REPO_MARK = WORK / "rivwq_repo_commit"
SESSION_MARK = WORK / "rivwq_session_start"
KEEPALIVE_SCRIPT = WORK / "rivwq_keepalive.py"
PYTHON = REPO / ".venv" / "bin" / "python"


def ensure_drive():
    if not (MOUNT / "MyDrive").is_dir():
        drive.mount(str(MOUNT))
    DRIVE_RUN.parent.mkdir(parents=True, exist_ok=True)
    DRIVE_LOGS.mkdir(parents=True, exist_ok=True)


def utc_now():
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())


def run(command, cwd=None, env=None):
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait() != 0:
        raise RuntimeError(f"exit code {process.returncode} from: {command[0]}")


def capture(command, cwd=None):
    done = subprocess.run(
        [str(part) for part in command], cwd=cwd, capture_output=True, text=True, check=False
    )
    if done.returncode != 0:
        raise RuntimeError(f"exit code {done.returncode} from {command[0]}: {done.stderr[-2000:]}")
    return done.stdout


def pid_running(pid_file):
    if not pid_file.exists():
        return False
    text = pid_file.read_text(encoding="utf-8").strip()
    stat = Path("/proc") / text / "stat"
    if not text.isdigit() or not stat.exists():
        return False
    return stat.read_text(encoding="utf-8").rsplit(")", 1)[1].split()[0] != "Z"


ensure_drive()
if not SESSION_MARK.exists():
    SESSION_MARK.write_text(str(time.time()), encoding="utf-8")
SESSION_START = float(SESSION_MARK.read_text(encoding="utf-8"))
SESSION_TAG = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime(SESSION_START))
DEADLINE = SESSION_START + SESSION_LIMIT_HOURS * 3600 - STOP_MARGIN_MINUTES * 60
uptime_hours = float(Path("/proc/uptime").read_text(encoding="utf-8").split()[0]) / 3600
print(f"session {SESSION_TAG}, machine uptime {uptime_hours:.2f} h")
print(f"automatic stop in {(DEADLINE - time.time()) / 3600:.2f} h")

In [ ]:
def ask_commit():
    commit = input("commit hash (40 hex characters): ").strip().lower()
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("type the full 40-character commit hash")
    return commit


def register_repo(commit):
    on_disk = REPO_MARK.read_text(encoding="utf-8").strip() if REPO_MARK.exists() else commit
    if on_disk != commit:
        raise RuntimeError(f"the repo on this disk is at {on_disk}; delete the runtime to change")
    recorded = commit
    if COMMIT_RECORD.exists():
        recorded = COMMIT_RECORD.read_text(encoding="utf-8").strip()
    if recorded != commit and not ALLOW_COMMIT_CHANGE:
        raise RuntimeError(f"run {RUN_ID} used commit {recorded} before; refusing {commit}")
    REPO_MARK.write_text(commit, encoding="utf-8")
    COMMIT_RECORD.write_text(commit + "\n", encoding="utf-8")
    print(f"repo ready at {REPO}, commit {commit}")


if REPO_SOURCE == "git":
    wanted = ask_commit()
    if not (REPO / ".git").exists():
        run(["git", "clone", "--no-checkout", REPO_URL, REPO])
        run(["git", "checkout", "--detach", wanted], cwd=REPO)
    head = capture(["git", "rev-parse", "HEAD"], cwd=REPO).strip()
    changed = capture(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO).strip()
    if head != wanted or changed:
        raise RuntimeError(f"clone is at {head} with changes: {changed or 'none'}")
    register_repo(head)

In [ ]:
if REPO_SOURCE == "zip":
    wanted = ask_commit()
    with zipfile.ZipFile(REPO_ZIP) as archive:
        embedded = archive.comment.decode("ascii").strip()
        if embedded != wanted:
            raise RuntimeError(f"{REPO_ZIP.name} holds commit {embedded or 'unknown'}")
        if any(not name.startswith("rivwq/") for name in archive.namelist()):
            raise RuntimeError("the zip must have one top folder named rivwq")
        if not REPO.exists():
            archive.extractall(REPO.parent)
    register_repo(wanted)

In [ ]:
if shutil.which("uv") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
uv_env = {name: value for name, value in os.environ.items() if not name.startswith("UV_")}
run(["uv", "python", "install", PYTHON_VERSION], env=uv_env)
run(["uv", "sync", "--frozen", "--python", PYTHON_VERSION], cwd=REPO, env=uv_env)
SETTINGS_CODE = (
    "import json, sys, ee; from rivwq.gee import export; config = export.load_config();"
    " print(json.dumps({'python': sys.version.split()[0], 'earthengine_api': ee.__version__,"
    " 'lock_name': export.LOCK_NAME, 'raw_root': config['raw_root'],"
    " 'log_dir': config['log_dir'], 'workers': config['workers'],"
    " 'project_variable': config['project_environment_variable']}))"
)
PROJECT = json.loads(capture([PYTHON, "-c", SETTINGS_CODE], cwd=REPO))
print(PROJECT)

LOCK_NAME = PROJECT["lock_name"]
RUN_DIR = REPO / PROJECT["raw_root"] / RUN_ID
LOG_DIR = REPO / PROJECT["log_dir"]
PID_FILE = LOG_DIR / f"{RUN_ID}.pid"
RUN_LOG = LOG_DIR / f"{RUN_ID}.log"
KEEPALIVE_PID_FILE = LOG_DIR / f"{RUN_ID}.keepalive.pid"
KEEPALIVE_LOG = LOG_DIR / f"{RUN_ID}.keepalive.log"
STOP_FLAG = LOG_DIR / f"{RUN_ID}.stop"
LOG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import getpass

import ee

ee.Authenticate(auth_mode=EE_AUTH_MODE, force=EE_AUTH_MODE is not None)
typed_project = getpass.getpass("Earth Engine project id (hidden): ").strip()
if not typed_project:
    raise ValueError("the project id is empty")
os.environ[PROJECT["project_variable"]] = typed_project
ee.Initialize(project=typed_project)
del typed_project
SESSION_CHECK = (
    "from rivwq.gee import session; session.initialize();"
    " print('the project environment reaches Earth Engine')"
)
run([PYTHON, "-c", SESSION_CHECK], cwd=REPO)

In [ ]:
TEMPORARY_SUFFIXES = (".part", ".syncing")
WRITTEN_LAST = ("manifest.json", "run_state.json")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def is_complete_file(path):
    is_temporary = path.suffix in TEMPORARY_SUFFIXES or path.name == LOCK_NAME
    return path.is_file() and not is_temporary


def complete_files(folder):
    found = [path for path in sorted(folder.rglob("*")) if is_complete_file(path)]
    return {path.relative_to(folder).as_posix(): path for path in found}


def copy_atomic(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + ".syncing")
    shutil.copyfile(source, temporary)
    os.replace(temporary, target)


def is_current(source, target):
    if not target.exists():
        return False
    same_size = target.stat().st_size == source.stat().st_size
    return same_size and target.stat().st_mtime >= source.stat().st_mtime


def sync_run_folder(source, target):
    last = {name: (source / name).read_bytes() for name in WRITTEN_LAST}
    copied = 0
    for name, path in complete_files(source).items():
        if name not in WRITTEN_LAST and not is_current(path, target / name):
            copy_atomic(path, target / name)
            copied += 1
    for name, content in last.items():
        temporary = target / (name + ".syncing")
        temporary.parent.mkdir(parents=True, exist_ok=True)
        temporary.write_bytes(content)
        os.replace(temporary, target / name)
    return copied


def entry_matches(folder, entry):
    path = folder / entry["path"]
    if not path.is_file() or path.stat().st_size != entry["bytes"]:
        return False
    return file_sha256(path) == entry["sha256"]


def verify_run_folder(folder):
    manifest = read_json(folder / "manifest.json")
    state = read_json(folder / "run_state.json")
    bad = sorted(item["path"] for item in manifest["files"] if not entry_matches(folder, item))
    valid = {Path(item["path"]).stem for item in manifest["files"] if item["path"] not in bad}
    unbacked = sorted(name for name in state["done"] if name not in valid)
    leftovers = sorted(
        path.relative_to(folder).as_posix()
        for path in folder.rglob("*")
        if path.is_file() and not is_complete_file(path)
    )
    print(f"{folder}")
    print(f"  manifest files matching sha256 and size: {len(valid)} of {len(manifest['files'])}")
    print(f"  site-years done {len(state['done'])}, pending {len(state['pending'])}")
    print(f"  bad files {bad}, done without a valid file {unbacked}, not copied {leftovers}")
    return {"ok": not bad and not unbacked, "bad": bad, "leftovers": leftovers}


def restore_run_folder():
    if (RUN_DIR / "manifest.json").exists():
        print("the run folder is already on this disk and is kept as it is")
        return
    if (DRIVE_RUN / "manifest.json").exists():
        copied = sync_run_folder(DRIVE_RUN, RUN_DIR)
        print(f"restored {copied} files plus manifest and state from {DRIVE_RUN}")
        return
    print(f"first session: unpacking {RUN_ZIP.name}, sha256 {file_sha256(RUN_ZIP)}")
    with zipfile.ZipFile(RUN_ZIP) as archive:
        names = [name for name in archive.namelist() if not name.endswith("/")]
        if any(not name.startswith(f"{RUN_ID}/") for name in names):
            raise RuntimeError(f"the zip must have one top folder named {RUN_ID}")
        skipped = [name for name in names if name.endswith((*TEMPORARY_SUFFIXES, LOCK_NAME))]
        archive.extractall(RUN_DIR.parent, members=[n for n in names if n not in skipped])
    print(f"unpacked {len(names) - len(skipped)} files, left out {skipped}")


def sync_to_drive():
    ensure_drive()
    copied = sync_run_folder(RUN_DIR, DRIVE_RUN)
    for log in (RUN_LOG, KEEPALIVE_LOG):
        if log.exists():
            copy_atomic(log, DRIVE_LOGS / f"{SESSION_TAG}_{log.name}")
    print(f"{utc_now()} synced to Drive, {copied} new or changed files")


def show_status():
    run(["make", "-s", "gee-status", f"RUN_ID={RUN_ID}", f"YEARS={YEARS}"], cwd=REPO)


def check_run_folder():
    checked = verify_run_folder(RUN_DIR)
    if not checked["ok"] and not ALLOW_BAD_FILES:
        raise RuntimeError("files do not match the manifest; see the lines above")
    manifest = read_json(RUN_DIR / "manifest.json")
    plan = json.loads(capture([PYTHON, "pipeline/roi_plan.py", "--json"], cwd=REPO))
    compared = {
        "run id": (manifest["run_id"], RUN_ID),
        "ROI hash": (manifest["roi_hash"], plan["roi_hash_with_skips"]),
        "ROI flags sha256": (manifest["roi_flags_sha256"], plan["flags_sha256"]),
    }
    for label, (stored, expected) in compared.items():
        print(f"  {label}: manifest {stored}, this repo {expected}")
        if stored != expected:
            raise RuntimeError(f"{label} of the run folder differs from this repo")


restore_run_folder()
check_run_folder()
show_status()
sync_to_drive()

In [ ]:
%%writefile /content/rivwq_keepalive.py
import argparse
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path


def log(message):
    print(f"{datetime.now(UTC):%Y-%m-%dT%H:%M:%SZ} keepalive: {message}", flush=True)


def stop_requested_within(seconds, stop_flag):
    end = time.monotonic() + seconds
    while time.monotonic() < end and not stop_flag.exists():
        time.sleep(1)
    return stop_flag.exists()


def run_once(command, log_path, pid_file):
    with open(log_path, "a", encoding="utf-8") as handle:
        child = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT)
        pid_file.write_text(str(child.pid), encoding="utf-8")
        log(f"started pid {child.pid}")
        code = child.wait()
    log(f"pid {child.pid} ended with exit code {code}")
    return code


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pid-file", type=Path, required=True)
    parser.add_argument("--stop-flag", type=Path, required=True)
    parser.add_argument("--log", type=Path, required=True)
    parser.add_argument("--wait-seconds", type=float, required=True)
    parser.add_argument("--max-restarts", type=int, required=True)
    parser.add_argument("command", nargs="+")
    args = parser.parse_args()
    restarts = 0
    while not args.stop_flag.exists():
        code = run_once(args.command, args.log, args.pid_file)
        if code == 0:
            log("exit code 0, nothing to restart")
            return 0
        if args.stop_flag.exists():
            break
        if restarts >= args.max_restarts:
            log(f"giving up after {restarts} restarts")
            return 1
        restarts += 1
        log(f"restart {restarts} of {args.max_restarts} in {args.wait_seconds:.0f} s")
        if stop_requested_within(args.wait_seconds, args.stop_flag):
            break
    log("stopped on request")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
EXTRACTION = [PYTHON, "pipeline/02_extract_gee.py", "--mode", "full", "--run-id", RUN_ID]
EXTRACTION += ["--years", YEARS]


def extraction_running():
    lock_held = pid_running(RUN_DIR / LOCK_NAME)
    return pid_running(KEEPALIVE_PID_FILE) or pid_running(PID_FILE) or lock_held


def start_keepalive():
    if extraction_running():
        print("already running on this machine; nothing started")
        return
    if time.time() >= DEADLINE:
        raise RuntimeError("this session is past its automatic stop time; open a new session")
    if not os.environ.get(PROJECT["project_variable"]):
        raise RuntimeError("the project id is not set in this session; run cell 4 first")
    STOP_FLAG.unlink(missing_ok=True)
    options = ["--pid-file", PID_FILE, "--stop-flag", STOP_FLAG, "--log", RUN_LOG]
    options += ["--wait-seconds", RESTART_WAIT_SECONDS, "--max-restarts", MAX_RESTARTS, "--"]
    command = [PYTHON, KEEPALIVE_SCRIPT, *options, *EXTRACTION]
    with open(KEEPALIVE_LOG, "a", encoding="utf-8") as handle:
        keepalive = subprocess.Popen(
            [str(part) for part in command],
            cwd=REPO,
            stdin=subprocess.DEVNULL,
            stdout=handle,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    KEEPALIVE_PID_FILE.write_text(str(keepalive.pid), encoding="utf-8")
    print(f"{utc_now()} keepalive started, pid {keepalive.pid}, years {YEARS}")


def graceful_stop():
    STOP_FLAG.touch()
    end = time.time() + STOP_TIMEOUT_SECONDS
    while extraction_running() and time.time() < end:
        for holder in (PID_FILE, RUN_DIR / LOCK_NAME):
            if pid_running(holder):
                os.kill(int(holder.read_text(encoding="utf-8").strip()), signal.SIGTERM)
        time.sleep(1)
    stopped = not extraction_running()
    print(f"{utc_now()} extraction stopped: {stopped}")
    return stopped


def verify_drive_copy():
    ensure_drive()
    checked = verify_run_folder(DRIVE_RUN)
    local = complete_files(RUN_DIR)
    differing = sorted(
        name
        for name, path in local.items()
        if not (DRIVE_RUN / name).is_file() or file_sha256(DRIVE_RUN / name) != file_sha256(path)
    )
    print(f"  files identical on this disk and on Drive: {len(local) - len(differing)}")
    print(f"  files differing or missing on Drive: {differing}")
    verified = checked["ok"] and not checked["leftovers"] and not differing
    print(f"{utc_now()} Drive copy verified: {verified}")
    return verified


def finish_session(reason):
    print(f"{utc_now()} {reason}")
    graceful_stop()
    sync_to_drive()
    verified = verify_drive_copy()
    drive.flush_and_unmount()
    print(f"{utc_now()} Drive flushed; safe to close this session: {verified}")
    return verified


start_keepalive()

In [ ]:
from IPython.display import clear_output


def report_progress():
    clear_output(wait=True)
    print(f"{utc_now()} automatic stop in {(DEADLINE - time.time()) / 3600:.2f} h")
    try:
        show_status()
    except RuntimeError as error:
        print(f"status failed: {error}")
    if KEEPALIVE_LOG.exists():
        print("".join(KEEPALIVE_LOG.read_text(encoding="utf-8").splitlines(keepends=True)[-3:]))


def sync_or_report():
    try:
        sync_to_drive()
    except OSError as error:
        print(f"{utc_now()} sync failed, will try again: {error}")


def monitor():
    next_status = 0.0
    next_sync = time.time() + SYNC_EVERY_MINUTES * 60
    while True:
        now = time.time()
        if now >= DEADLINE:
            return finish_session("automatic stop before the session limit")
        if not pid_running(KEEPALIVE_PID_FILE):
            report_progress()
            return finish_session("the keepalive loop has ended")
        if now >= next_status:
            report_progress()
            next_status = now + STATUS_EVERY_MINUTES * 60
        if now >= next_sync:
            sync_or_report()
            next_sync = now + SYNC_EVERY_MINUTES * 60
        time.sleep(MONITOR_TICK_SECONDS)


try:
    monitor()
except KeyboardInterrupt:
    print(f"{utc_now()} monitor interrupted; the extraction keeps running, nothing is synced")

In [ ]:
finish_session("stop requested by hand")

In [ ]:
if not verify_drive_copy() and not extraction_running():
    raise RuntimeError("the Drive copy is not verified; do not delete this runtime yet")
show_status()
drive.flush_and_unmount()